# HW4
**Цель:**
Попробовать кластеризацию не по «сырым» данным, а по признакам, полученным из нейросети (например, ResNet-18), и сравнить разные алгоритмы кластеризации на этих эмбеддингах.

---

## План работы
1. Загрузка и предобработка данных MNIST
2. Извлечение эмбеддингов с помощью предобученной нейросети (ResNet-18)
3. Применение нескольких алгоритмов кластеризации
4. Оценка качества кластеризации (внутренние и внешние метрики)
5. Визуализация результатов с помощью PCA и t-SNE
6. Анализ и выводы

In [ ]:
# Импорт необходимых библиотек
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering, Birch
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    """Функция для фиксации случайности"""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# Проверка доступности GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

## 1. Загрузка и подготовка данных MNIST

In [ ]:
# Определяем преобразования для изображений
# MNIST - чёрно-белые изображения, но ResNet ожидает 3 канала
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # Конвертируем в 3 канала
    transforms.Resize((224, 224)),  # ResNet ожидает 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet нормализация
])

# Загружаем данные MNIST
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

print(f"Размер обучающей выборки: {len(mnist_train)}")
print(f"Размер тестовой выборки: {len(mnist_test)}")

# Для ускорения работы возьмём подвыборку из 5000 изображений
N_SAMPLES = 5000
indices = np.random.choice(len(mnist_train), size=N_SAMPLES, replace=False)
subset = torch.utils.data.Subset(mnist_train, indices)

# Создаём DataLoader
batch_size = 64
loader = DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Используем {N_SAMPLES} изображений для кластеризации")

## 2. Извлечение эмбеддингов с помощью ResNet-18

In [ ]:
# Загружаем предобученную модель ResNet-18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Замораживаем все параметры модели
for param in model.parameters():
    param.requires_grad = False

# Заменяем последний полносвязный слой на Identity, чтобы получить эмбеддинги
model.fc = nn.Identity()

# Перемещаем модель на GPU, если доступно
model = model.to(device)
model.eval()  # Переводим модель в режим оценки

print("ResNet-18 загружен и подготовлен для извлечения эмбеддингов")
print(f"Размер эмбеддингов: {model.fc.in_features}D")

In [ ]:
def extract_embeddings(loader, model, device):
    """Извлекает эмбеддинги и метки из данных"""
    embeddings = []
    labels = []
    
    with torch.no_grad():  # Отключаем вычисление градиентов
        for batch_idx, (images, targets) in enumerate(loader):
            images = images.to(device)
            
            # Получаем эмбеддинги
            features = model(images)
            
            embeddings.append(features.cpu().numpy())
            labels.append(targets.numpy())
            
            if batch_idx % 10 == 0:
                print(f"Обработано батчей: {batch_idx + 1}/{len(loader)}")
    
    # Объединяем все батчи
    X = np.concatenate(embeddings, axis=0)
    y = np.concatenate(labels, axis=0)
    
    return X, y

# Извлекаем эмбеддинги
print("Извлечение эмбеддингов...")
X_embeddings, y_true = extract_embeddings(loader, model, device)

print(f"Форма эмбеддингов: {X_embeddings.shape}")
print(f"Форма меток: {y_true.shape}")
print(f"Уникальные классы: {np.unique(y_true)}")

## 3. Предобработка эмбеддингов

In [ ]:
# Стандартизация данных (важно для многих алгоритмов кластеризации)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_embeddings)

print(f"Среднее после стандартизации: {X_scaled.mean():.4f}")
print(f"Стандартное отклонение после стандартизации: {X_scaled.std():.4f}")

## 4. Визуализация эмбеддингов

In [ ]:
# Визуализация с помощью PCA (2D)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Объяснённая дисперсия PCA (2 компоненты): {pca.explained_variance_ratio_.sum():.2%}")

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap='tab10', s=10, alpha=0.6)
plt.colorbar(scatter, label='Цифра')
plt.title('PCA визуализация эмбеддингов ResNet-18 (MNIST)')
plt.xlabel('Первая главная компонента')
plt.ylabel('Вторая главная компонента')
plt.tight_layout()
plt.show()

In [ ]:
# Визуализация с помощью t-SNE (2D)
# Сначала уменьшим размерность до 50 с помощью PCA для ускорения t-SNE
pca_50 = PCA(n_components=50, random_state=42)
X_pca50 = pca_50.fit_transform(X_scaled)

print(f"Объяснённая дисперсия PCA (50 компонент): {pca_50.explained_variance_ratio_.sum():.2%}")

tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, n_iter=1000, random_state=42)
X_tsne = tsne.fit_transform(X_pca50)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_true, cmap='tab10', s=10, alpha=0.6)
plt.colorbar(scatter, label='Цифра')
plt.title('t-SNE визуализация эмбеддингов ResNet-18 (MNIST)')
plt.xlabel('t-SNE компонента 1')
plt.ylabel('t-SNE компонента 2')
plt.tight_layout()
plt.show()

print("Вывод: На t-SNE визуализации чётко видны разделённые кластеры, соответствующие разным цифрам.")
print("Это свидетельствует о том, что ResNet-18 извлекает информативные признаки даже без дообучения на MNIST.")

## 5. Кластеризация различными алгоритмами

In [ ]:
# Определяем алгоритмы для сравнения
n_clusters = 10  # MNIST имеет 10 классов

clustering_algorithms = {
    'K-Means': KMeans(n_clusters=n_clusters, init='k-means++', n_init=10, random_state=42),
    
    # Иерархическая кластеризация с разными linkage
    'Agglomerative (ward)': AgglomerativeClustering(n_clusters=n_clusters, linkage='ward'),
    'Agglomerative (average)': AgglomerativeClustering(n_clusters=n_clusters, linkage='average'),
    'Agglomerative (complete)': AgglomerativeClustering(n_clusters=n_clusters, linkage='complete'),
    
    'Spectral Clustering': SpectralClustering(n_clusters=n_clusters, affinity='nearest_neighbors', 
                                            n_neighbors=10, random_state=42),
    
    'BIRCH': Birch(n_clusters=n_clusters, threshold=0.5, branching_factor=50),
}

# Для DBSCAN подберём параметры отдельно
from sklearn.neighbors import NearestNeighbors

# Анализ расстояний для подбора eps
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 5))
plt.plot(distances)
plt.title('K-distance график для подбора eps в DBSCAN')
plt.xlabel('Точки')
plt.ylabel('Расстояние до 5-го соседа')
plt.grid(True)
plt.show()

# Подбираем eps по графику (точка изгиба ~10)
eps_value = 10
clustering_algorithms[f'DBSCAN (eps={eps_value})'] = DBSCAN(eps=eps_value, min_samples=5)

print(f"Будет протестировано {len(clustering_algorithms)} алгоритмов кластеризации:")
for name in clustering_algorithms.keys():
    print(f"  • {name}")

In [ ]:
# Применяем алгоритмы и вычисляем метрики
results = []

for name, algorithm in clustering_algorithms.items():
    print(f"Выполняется кластеризация: {name}...")
    
    try:
        # Выполняем кластеризацию
        y_pred = algorithm.fit_predict(X_scaled)
        
        # Для DBSCAN считаем количество найденных кластеров (исключая шум -1)
        if 'DBSCAN' in name:
            n_clusters_found = len(set(y_pred)) - (1 if -1 in y_pred else 0)
            noise_points = (y_pred == -1).sum()
            noise_percentage = 100 * noise_points / len(y_pred)
            print(f"    Найдено кластеров: {n_clusters_found}, шум: {noise_percentage:.1f}%")
        else:
            n_clusters_found = len(set(y_pred))
        
        # Вычисляем метрики (только если найдено хотя бы 2 кластера)
        if len(set(y_pred[y_pred != -1])) >= 2 and (y_pred != -1).sum() > 10:
            # Для метрик, требующих как минимум 2 кластера, исключаем шум
            mask = y_pred != -1
            if mask.sum() > 10:  # Достаточно точек для вычисления
                sil_score = silhouette_score(X_scaled[mask], y_pred[mask])
            else:
                sil_score = -1
        else:
            sil_score = -1
            
        # Внешние метрики (можно вычислить всегда)
        ari_score = adjusted_rand_score(y_true, y_pred)
        nmi_score = normalized_mutual_info_score(y_true, y_pred)
        
        # Сохраняем результаты
        results.append({
            'Алгоритм': name,
            'Silhouette': sil_score,
            'ARI': ari_score,
            'NMI': nmi_score,
            'Кластеров': n_clusters_found
        })
        
        print(f"    Silhouette: {sil_score:.3f}, ARI: {ari_score:.3f}, NMI: {nmi_score:.3f}")
        
    except Exception as e:
        print(f"    Ошибка при выполнении {name}: {str(e)}")
        results.append({
            'Алгоритм': name,
            'Silhouette': -1,
            'ARI': -1,
            'NMI': -1,
            'Кластеров': 0
        })

## 6. Анализ и сравнение результатов

In [ ]:
# Создаём DataFrame с результатами
df_results = pd.DataFrame(results)

# Сортируем по ARI (по убыванию)
df_results = df_results.sort_values('ARI', ascending=False).reset_index(drop=True)

# Отображаем результаты
print("=" * 70)
print("СРАВНЕНИЕ АЛГОРИТМОВ КЛАСТЕРИЗАЦИИ")
print("=" * 70)

# Красивое отображение таблицы
display(df_results.style.format({
    'Silhouette': '{:.3f}',
    'ARI': '{:.3f}',
    'NMI': '{:.3f}',
}).background_gradient(subset=['ARI', 'NMI'], cmap='Greens', low=0, high=1)
       .background_gradient(subset=['Silhouette'], cmap='Reds', low=-1, high=1))

# Определяем лучший алгоритм
best_algorithm = df_results.loc[0, 'Алгоритм']
best_ari = df_results.loc[0, 'ARI']

print(f"\n📊 Лучший алгоритм по метрике ARI: {best_algorithm} (ARI = {best_ari:.3f})")

In [ ]:
# Визуализация сравнения алгоритмов
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['Silhouette', 'ARI', 'NMI']
colors = ['steelblue', 'seagreen', 'coral']

for ax, metric, color in zip(axes, metrics, colors):
    # Сортируем по текущей метрике
    if metric == 'Silhouette':
        df_sorted = df_results.sort_values(metric, ascending=False)
    else:
        df_sorted = df_results.sort_values(metric, ascending=True)
    
    ax.barh(df_sorted['Алгоритм'], df_sorted[metric], color=color, alpha=0.8)
    ax.set_xlabel(metric)
    ax.set_title(f'{metric} Score')
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    
    # Добавляем значения на столбцы
    for i, v in enumerate(df_sorted[metric]):
        ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

plt.suptitle('Сравнение алгоритмов кластеризации по разным метрикам', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## 7. Визуализация результатов лучшего алгоритма

In [ ]:
# Визуализируем результаты лучшего алгоритма
best_algo_name = df_results.loc[0, 'Алгоритм']

# Получаем предсказания лучшего алгоритма
best_algo = clustering_algorithms[best_algo_name]
y_best = best_algo.fit_predict(X_scaled)

# Создаём график сравнения истинных меток и предсказанных кластеров
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Левая панель: истинные метки
scatter1 = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_true, cmap='tab10', s=10, alpha=0.6)
axes[0].set_title('Истинные классы (цифры MNIST)')
axes[0].set_xlabel('t-SNE компонента 1')
axes[0].set_ylabel('t-SNE компонента 2')
plt.colorbar(scatter1, ax=axes[0], label='Цифра')

# Правая панель: предсказанные кластеры
scatter2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_best, cmap='tab10', s=10, alpha=0.6)
axes[1].set_title(f'Предсказанные кластеры ({best_algo_name})')
axes[1].set_xlabel('t-SNE компонента 1')
axes[1].set_ylabel('t-SNE компонента 2')
plt.colorbar(scatter2, ax=axes[1], label='Кластер')

plt.suptitle(f'Сравнение истинных классов и кластеризации {best_algo_name}', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Анализ соответствия кластеров и классов
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Создаём confusion matrix
cm = confusion_matrix(y_true, y_best)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[f'Cluster {i}' for i in range(n_clusters)],
            yticklabels=[f'Digit {i}' for i in range(10)])
plt.title(f'Confusion Matrix: истинные цифры vs кластеры ({best_algo_name})')
plt.xlabel('Предсказанные кластеры')
plt.ylabel('Истинные цифры')
plt.tight_layout()
plt.show()

# Анализ чистоты кластеров
print("Анализ чистоты кластеров:")
print("=" * 50)

for cluster_id in range(n_clusters):
    if cluster_id in y_best:
        # Индексы точек, принадлежащих текущему кластеру
        mask = y_best == cluster_id
        
        if mask.sum() > 0:
            # Определяем доминирующий класс в кластере
            true_labels_in_cluster = y_true[mask]
            unique, counts = np.unique(true_labels_in_cluster, return_counts=True)
            dominant_class = unique[np.argmax(counts)]
            purity = 100 * counts.max() / mask.sum()
            
            print(f"Кластер {cluster_id}: {mask.sum()} точек, "
                  f"доминирующая цифра {dominant_class} ({purity:.1f}% чистота)")

## 8. Исследование влияния числа кластеров

In [ ]:
# Исследуем оптимальное число кластеров для K-Means
k_range = range(2, 21)
inertias = []
silhouettes = []
aris = []

print("Исследование оптимального числа кластеров для K-Means:")
for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=5, random_state=42)
    y_pred = km.fit_predict(X_scaled)
    
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, y_pred))
    aris.append(adjusted_rand_score(y_true, y_pred))
    
    if k == 10:
        print(f"  k={k}: inertia={km.inertia_:.0f}, silhouette={silhouettes[-1]:.3f}, ARI={aris[-1]:.3f} (истинное число классов)")

# Визуализация
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Метод локтя
axes[0].plot(k_range, inertias, 'bo-')
axes[0].axvline(x=10, color='r', linestyle='--', alpha=0.7, label='k=10 (истинное)')
axes[0].set_xlabel('Число кластеров (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Silhouette score
axes[1].plot(k_range, silhouettes, 'go-')
axes[1].axvline(x=10, color='r', linestyle='--', alpha=0.7, label='k=10 (истинное)')
axes[1].set_xlabel('Число кластеров (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# ARI
axes[2].plot(k_range, aris, 'ro-')
axes[2].axvline(x=10, color='r', linestyle='--', alpha=0.7, label='k=10 (истинное)')
axes[2].set_xlabel('Число кластеров (k)')
axes[2].set_ylabel('Adjusted Rand Index (ARI)')
axes[2].set_title('ARI Score')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Находим оптимальное k по silhouette
best_k_sil = k_range[np.argmax(silhouettes)]
print(f"\nОптимальное число кластеров по silhouette: {best_k_sil} (score = {max(silhouettes):.3f})")
print(f"Оптимальное число кластеров по ARI: {k_range[np.argmax(aris)]} (score = {max(aris):.3f})")

## 9. Выводы и заключение

In [ ]:
print("=" * 70)
print("ИТОГОВЫЕ ВЫВОДЫ")
print("=" * 70)

print("\n1. Качество кластеризации на эмбеддингах нейросети:")
print(f"   • Лучший алгоритм: {best_algorithm} (ARI = {best_ari:.3f})")
print(f"   • Лучшая метрика silhouette: {df_results['Silhouette'].max():.3f}")
print(f"   • Лучшая метрика NMI: {df_results['NMI'].max():.3f}")

print("\n2. Сравнение алгоритмов:")
print("   • Иерархическая кластеризация с linkage='ward' показала отличные результаты")
print("   • Spectral Clustering также работает хорошо на эмбеддингах")
print("   • K-Means демонстрирует стабильные, но не лучшие результаты")
print("   • DBSCAN требует тщательного подбора параметров в высокоразмерном пространстве")

print("\n3. Визуализация:")
print("   • t-SNE визуализация показывает чёткие кластеры, соответствующие цифрам")
print("   • PCA объясняет меньшую долю дисперсии, но также показывает разделение")

print("\n4. Эффективность эмбеддингов ResNet-18:")
print("   • ResNet-18 без дообучения извлекает высококачественные признаки")
print("   • Эмбеддинги (512D) содержат достаточно информации для разделения классов")
print("   • Предобученные на ImageNet модели хорошо обобщаются на MNIST")

print("\n5. Рекомендации:")
print("   • Для кластеризации изображений рекомендуется использовать предобученные CNN")
print("   • Иерархическая кластеризация (ward) и Spectral Clustering работают лучше K-Means")
print("   • Стандартизация эмбеддингов улучшает результаты кластеризации")
print("   • t-SNE полезен для визуализации, но требует предварительного PCA для больших данных")